# Helmet Compliance Detector — Colab Training Notebook

Trains both the YOLOv8 baseline and the YOLOv8+CBAM variant on a free-tier Colab GPU (T4).

This notebook uses the **hybrid workflow**: the dataset was already downloaded and
converted to YOLO format locally (`data/prepare_dataset.py`), zipped, and uploaded to
Google Drive as `data_dataset.zip` — Colab just unzips it, no Kaggle credentials needed
here. Steps: mount Drive -> get project code -> install deps -> unzip dataset -> train
baseline -> train CBAM -> compare.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/helmet-detection'
CODE_DIR = PROJECT_DIR + '/od'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)

## Get the project code

Cloned into its own subfolder (`CODE_DIR`, i.e. `PROJECT_DIR/od`) so it doesn't matter
whether `data_dataset.zip` or anything else is already sitting in `PROJECT_DIR` — public
repo, no credentials needed. Re-running this notebook in a later session detects the
existing clone and `git pull`s instead of trying to clone again.

In [ ]:
import os
if os.path.isdir(f"{CODE_DIR}/.git"):
    print("Repo already cloned, pulling latest instead...")
    !cd {CODE_DIR} && git pull
else:
    !git clone https://github.com/rozengoza/od.git {CODE_DIR}
%cd {CODE_DIR}
!pip install -q -r requirements.txt

## Get the prepared dataset

Upload `data_dataset.zip` (created locally by `python -m zipfile -c data_dataset.zip
data/dataset`) directly into `PROJECT_DIR` on Drive — i.e.
`/content/drive/MyDrive/helmet-detection/data_dataset.zip`, *next to* the `od` code
folder, not inside it. Unzipped here into the code folder's `data/dataset`. This avoids
re-downloading from Kaggle inside Colab entirely.

In [ ]:
import os
zip_path = f"{PROJECT_DIR}/data_dataset.zip"
assert os.path.exists(zip_path), \
    f'{zip_path} not found — upload data_dataset.zip to PROJECT_DIR on Drive first.'
!python -m zipfile -e {zip_path} .
!ls data/dataset && cat data/dataset/data.yaml

### (Alternative) download fresh from Kaggle instead

Only needed if you didn't prepare the dataset locally. Uncomment and run instead of the
unzip cell above.

In [ ]:
# from google.colab import files
# import os
# os.makedirs('/root/.kaggle', exist_ok=True)
# uploaded = files.upload()  # select kaggle.json
# for fname in uploaded:
#     os.rename(fname, '/root/.kaggle/kaggle.json')
# os.chmod('/root/.kaggle/kaggle.json', 0o600)
# !python data/prepare_dataset.py --out data/dataset --val-frac 0.1 --test-frac 0.1

## Train baseline YOLOv8s

In [ ]:
!python train.py --variant baseline --data data/dataset/data.yaml --model-size s --epochs 60 --imgsz 640 --batch 16

## Train YOLOv8s + CBAM (novel-method variant)

In [ ]:
!python train.py --variant cbam --data data/dataset/data.yaml --model-size s --epochs 60 --imgsz 640 --batch 16

## Compare baseline vs CBAM on the held-out test split

In [ ]:
!python evaluate.py \
  --weights runs/detect/helmet-baseline-yolov8s/weights/best.pt runs/detect/helmet-cbam-yolov8s/weights/best.pt \
  --names baseline cbam \
  --data data/dataset/data.yaml \
  --out docs/results_comparison.csv

## (Optional) Quick sanity check on a sample image

In [ ]:
from ultralytics import YOLO
from models import register_modules  # only needed for the CBAM checkpoint
model = YOLO('runs/detect/helmet-cbam-yolov8s/weights/best.pt')
results = model.predict('data/dataset/test/images', save=True, conf=0.35)
print('Annotated predictions saved under runs/detect/predict*/')

## After training: bring weights back to your local machine

`runs/` lives under Drive's `CODE_DIR` (`PROJECT_DIR/od`) already, so both `best.pt`
files persist there automatically. Download
`runs/detect/helmet-baseline-yolov8s/weights/best.pt` and
`runs/detect/helmet-cbam-yolov8s/weights/best.pt` from the Drive web UI (or sync via the
Drive desktop app) into your local project's matching `runs/detect/.../weights/` paths —
that's what `evaluate.py` and `app/streamlit_app.py` expect locally.